# **Referring Image Segmentation (RIS)**

**Mục tiêu:** tạo một pipeline huấn luyện nhẹ có thể train được trên Colab, nhưng **suy luận (inference) chạy tốt trên CPU**, với độ chính xác **tiệm cận** mô hình LAVT thông qua:  
- **Sinh viên (student)**: `CLIPSeg` — mô hình phân đoạn theo văn bản gọn nhẹ, dễ suy luận trên CPU.  
- **Giáo viên (teacher)**: `LAVT` — dùng để **distillation** nhanh (ít epoch) để kéo accuracy của student tiến gần LAVT.  
- **Tối ưu CPU**: xuất **ONNX** + **quantization** để chạy nhanh trên máy không có GPU.



# Cài đặt các thư viện cần thiết

In [ ]:
import sys, subprocess, os, platform

def pip_install(pkgs):
    print("Installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["gdown>=4.7.1"])

base_pkgs = [
    "torch>=2.1.0",
    "torchvision>=0.16.0",
    "transformers>=4.41.0",
    "timm>=0.9.16",
    "accelerate>=0.26.0",
    "albumentations>=1.4.7",
    "opencv-python",
    "scikit-image",
    "matplotlib",
    "numpy",
    "pandas",
    "pycocotools",
    "einops",
    "optimum>=1.14.0",
    "pillow<11"
]
pip_install(base_pkgs)

import torch, transformers, timm, albumentations, cv2, numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("OpenCV:", cv2.__version__)


Installing: ['gdown>=4.7.1']
Installing: ['torch>=2.1.0', 'torchvision>=0.16.0', 'transformers>=4.41.0', 'timm>=0.9.16', 'accelerate>=0.26.0', 'albumentations>=1.4.7', 'opencv-python', 'scikit-image', 'matplotlib', 'numpy', 'pandas', 'pycocotools', 'einops', 'optimum>=1.14.0', 'pillow<11']
PyTorch: 2.8.0+cu126 | CUDA available: True
Transformers: 4.57.1
OpenCV: 4.12.0


## Cấu hình đường dẫn

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/referring-image-segmentation"
print("Mounted Google Drive. ROOT =", ROOT)

DATA        = f"{ROOT}/data"
REFMIX      = f"{DATA}/refmix"
CSV_RCOCO   = os.environ.get("CSV_RCOCO",   f"{DATA}/refcoco/annotations/refcoco.csv")
CSV_RPLUS   = os.environ.get("CSV_RPLUS",   f"{DATA}/refcoco_plus/annotations/refcoco_plus.csv")
CKPT_DIR = os.environ.get("CKPT_DIR", f"{ROOT}/ckpts/clipseg_student")

TRAIN_JSONL = os.environ.get("TRAIN_JSONL", f"{REFMIX}/train.jsonl")
VAL_RCOCO_JSONL = os.environ.get("VAL_RCOCO_JSONL", f"{REFMIX}/val_refcoco.jsonl")
VAL_RCOCO_PLUS_JSONL = os.environ.get("VAL_RCOCO_PLUS_JSONL", f"{REFMIX}/val_refcoco_plus.jsonl")

os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_LAST = os.path.join(CKPT_DIR, "ckpt_last.pt")
CKPT_BEST = os.path.join(CKPT_DIR, "ckpt_best.pt")
HF_BEST_DIR = os.path.join(CKPT_DIR, "hf_best")

Mounted at /content/drive
Mounted Google Drive. ROOT = /content/drive/MyDrive/referring-image-segmentation


# Cấu hình các tham số

In [ ]:
IMAGE_SIZE   = int(os.environ.get("IMAGE_SIZE", 448))
BATCH_SIZE   = int(os.environ.get("BATCH_SIZE", 18))
VAL_BS       = int(os.environ.get("VAL_BS", 8))
NUM_WORKERS  = int(os.environ.get("NUM_WORKERS", 2))
LR           = float(os.environ.get("LR", 1e-4))
WD           = float(os.environ.get("WEIGHT_DECAY", 1e-4))
EPOCHS       = 1
TARGET_STEPS = int(os.environ.get("TARGET_STEPS", 30000))
WARMUP_STEPS = int(os.environ.get("WARMUP_STEPS", 500))

VAL_SET = os.environ.get("VAL_SET", "both").strip().lower()
LOG_EVERY_STEPS   = int(os.environ.get("LOG_EVERY_STEPS", 100))
VAL_EVERY_STEPS   = int(os.environ.get("VAL_EVERY_STEPS", 500))
VAL_MAX_BATCHES   = int(os.environ.get("VAL_MAX_BATCHES", 200))
CKPT_EVERY_STEPS  = int(os.environ.get("CKPT_EVERY_STEPS", VAL_EVERY_STEPS))

# DataLoader

In [ ]:
import os, time
import numpy as np
import pandas as pd
import cv2, torch
from torch.utils.data import Dataset
import albumentations as A

# ---- IO helper (đọc có retry) ----
def safe_imread(path, flags, max_retries: int = 5, backoff: float = 0.6):
    for i in range(max_retries):
        if path and os.path.exists(path):
            img = cv2.imread(path, flags)
            if img is not None:
                return img
        time.sleep(backoff * (i + 1))
    return None

# ---- resize giữ tỉ lệ + CENTER pad ----
def resize_keep_ratio_and_pad(img, size: int, is_mask: bool = False):
    h, w = img.shape[:2] if img.ndim == 3 else img.shape
    scale = size / max(h, w) if max(h, w) > 0 else 1.0
    nh, nw = int(round(h * scale)), int(round(w * scale))
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    resized = cv2.resize(img, (nw, nh), interpolation=interp)

    pad_top    = (size - nh) // 2
    pad_bottom = size - nh - pad_top
    pad_left   = (size - nw) // 2
    pad_right  = size - nw - pad_left

    border_val = 0 if img.ndim == 2 else (0, 0, 0)
    out = cv2.copyMakeBorder(resized, pad_top, pad_bottom, pad_left, pad_right,
                             borderType=cv2.BORDER_CONSTANT, value=border_val)
    return out, (h, w)

class RefSegCSVDataset(Dataset):
    """
    CSV cần cột: image_path, mask_path, expr, split
    Output:
      - nếu return_torch=True:
          image: float32 [3,H,W] in [0,1], mask: float32 [1,H,W] {0,1}
      - nếu return_torch=False:
          image: uint8 HWC RGB,          mask: uint8 HW {0,1}
    """
    def __init__(
        self,
        csv_path: str,
        dataset_name: str = "refcoco",
        split: str = "train",
        image_size: int = 448,
        augment: bool = True,
        return_torch: bool = True,
    ):
        df = pd.read_csv(csv_path)
        need = {"image_path", "mask_path", "expr", "split"}
        miss = need - set(df.columns)
        if miss:
            raise ValueError(f"{csv_path} thiếu cột: {miss}. Cần đủ {need}")

        self.df = df[df["split"].astype(str).str.lower() == str(split).lower()].reset_index(drop=True)
        self.dataset_name = dataset_name
        self.size = int(image_size)
        self.return_torch = bool(return_torch)

        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.10, rotate_limit=10, p=0.5),
            A.ColorJitter(p=0.25),
        ]) if augment else A.Compose([])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        ipath = str(row["image_path"])
        mpath = "" if pd.isna(row["mask_path"]) else str(row["mask_path"])
        raw_t = row["expr"]
        text  = "" if (raw_t is None or (isinstance(raw_t, float) and np.isnan(raw_t))) else str(raw_t)

        # ---- read image ----
        img = safe_imread(ipath, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(ipath)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ---- read mask (nhị phân) ----
        if mpath and os.path.exists(mpath):
            m = safe_imread(mpath, cv2.IMREAD_GRAYSCALE)
            mask = (m > 127).astype(np.uint8) if m is not None else np.zeros(img.shape[:2], np.uint8)
        else:
            mask = np.zeros(img.shape[:2], np.uint8)

        # ---- augment ----
        out = self.aug(image=img, mask=mask)
        img, mask = out["image"], out["mask"]

        # ---- resize + center pad ----
        img_pad,  orig_hw = resize_keep_ratio_and_pad(img,  self.size, is_mask=False)
        mask_pad, _       = resize_keep_ratio_and_pad(mask, self.size, is_mask=True)

        if not self.return_torch:
            return {
                "image": img_pad.astype(np.uint8),         # HWC RGB
                "mask":  mask_pad.astype(np.uint8),        # HW {0,1}
                "text":  text,
                "dataset": self.dataset_name,
                "image_path": ipath,
                "has_mask": int(mask_pad.sum() > 0),
                "orig_size": orig_hw,
            }

        # to torch
        img_t = torch.from_numpy(np.ascontiguousarray(img_pad)).permute(2, 0, 1).float() / 255.0
        msk_t = (torch.from_numpy(np.ascontiguousarray(mask_pad)) > 0).to(torch.float32).unsqueeze(0)

        return {
            "image": img_t,   # [3,H,W] in [0,1]
            "mask":  msk_t,   # [1,H,W] in {0,1}
            "text":  text,
            "dataset": self.dataset_name,
            "image_path": ipath,
            "has_mask": int(mask_pad.sum() > 0),
            "orig_size": orig_hw,
        }

# ---- collate helpers ----
def collate_default(batch):
    from torch.utils.data.dataloader import default_collate
    return default_collate(batch)

def collate_identity(batch):
    return batch


In [ ]:
from torch.utils.data import ConcatDataset, DataLoader

train_rcc  = RefSegCSVDataset(CSV_RCOCO, dataset_name="refcoco",
                              split="train", image_size=IMAGE_SIZE,
                              augment=True,  return_torch=False)

train_rccp = RefSegCSVDataset(CSV_RPLUS, dataset_name="refcoco_plus",
                              split="train", image_size=IMAGE_SIZE,
                              augment=True,  return_torch=False)

val_rcc    = RefSegCSVDataset(CSV_RCOCO, dataset_name="refcoco",
                              split="val",   image_size=IMAGE_SIZE,
                              augment=False, return_torch=False)

val_rccp   = RefSegCSVDataset(CSV_RPLUS, dataset_name="refcoco_plus",
                              split="val",   image_size=IMAGE_SIZE,
                              augment=False, return_torch=False)



train_ds = ConcatDataset([train_rcc, train_rccp])
collate_identity = lambda batch: batch

# === Dataloaders ===
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_identity,
)

val_loader_rcc  = DataLoader(
    val_rcc, batch_size=VAL_BS, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_identity,
)

val_loader_rccp = DataLoader(
    val_rccp, batch_size=VAL_BS, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=collate_identity,
)

# === Chọn val loader theo VAL_SET ===
if VAL_SET == "refcoco":
    val_loader = val_loader_rcc
elif VAL_SET in ("refcoco_plus", "refcoco+","plus","rccp"):
    val_loader = val_loader_rccp
else:
    val_loader = None

# === Logs ===
print(f"[train] refcoco={len(train_rcc)} | refcoco_plus={len(train_rccp)} | total={len(train_ds)}")
print(f"[val]   refcoco={len(val_rcc)} | refcoco_plus={len(val_rccp)}")
print("ĐÁNH GIÁ TRÊN CẢ 2 TẬP RefCOCO và RefCOCO+" if val_loader is None else f"SẼ ĐÁNH GIÁ TRÊN: {VAL_SET}")


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


[train] refcoco=120624 | refcoco_plus=120191 | total=240815
[val]   refcoco=10834 | refcoco_plus=10758
ĐÁNH GIÁ TRÊN CẢ 2 TẬP RefCOCO và RefCOCO+



# Distillation từ **LAVT** để kéo accuracy

In [ ]:
USE_LAVT_DISTILL = True

# Thư mục & checkpoint
LAVT_DIR = "/content/LAVT-RIS"
LAVT_REFCOCO_CKPT_ID = "13D-OeEOijV8KTC3BkFP-gOJymc6DLwVT"
PSEUDO_DIR = "/content/pseudo_lavt"

import os, sys, subprocess, time, json, shutil, pandas as pd
from pathlib import Path

os.makedirs(PSEUDO_DIR, exist_ok=True)

def ensure_lavt_and_ckpt(lavt_dir, ckpt_id):
    if not os.path.exists(lavt_dir):
        subprocess.check_call(["git", "clone", "https://github.com/yz93/LAVT-RIS.git", lavt_dir])
    ckpt_dir = os.path.join(lavt_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "refcoco.pth")
    if not os.path.exists(ckpt_path):
        subprocess.check_call(["gdown", "--id", ckpt_id, "-O", ckpt_path])
        print("Đã tải checkpoint:", ckpt_path)
    return ckpt_path

def make_pseudo_dir(root, dataset_name):
    out = os.path.join(root, dataset_name)
    os.makedirs(out, exist_ok=True)
    return out

def run_lavt_inference(lavt_dir, ckpt_path, image_path, text, out_mask):
    cmd = [
        sys.executable, os.path.join(lavt_dir, "demo_inference.py"),
        "--image", image_path,
        "--text", text,
        "--resume", ckpt_path,
        "--save_mask", out_mask
    ]
    try:
        subprocess.run(cmd, check=True)
        return True
    except Exception as e:
        print(f"Bỏ qua 1 mẫu: {image_path} | Lỗi: {e}")
        return False


if USE_LAVT_DISTILL:
    ckpt_path = ensure_lavt_and_ckpt(LAVT_DIR, LAVT_REFCOCO_CKPT_ID)

Đã tải checkpoint: /content/LAVT-RIS/checkpoints/refcoco.pth


# Metric (mIoU, Precision@IoU=0.5)


In [ ]:
import torch

def iou_bin(pred, target, eps=1e-6):
    # pred, target: float or int tensor (B,H,W), values 0..1
    pred = (pred>0.5).float()
    target = (target>0.5).float()
    inter = (pred*target).sum(dim=(1,2))
    union = pred.sum(dim=(1,2)) + target.sum(dim=(1,2)) - inter + eps
    return (inter/union).mean().item()

def precision_at_threshold(pred, target, thr=0.5):
    # proportion of samples with IoU >= thr
    pred = (pred>0.5).float()
    target = (target>0.5).float()
    B = pred.shape[0]
    ious = []
    for b in range(B):
        inter = (pred[b]*target[b]).sum()
        union = pred[b].sum() + target[b].sum() - inter + 1e-6
        ious.append((inter/union).item())
    ious = torch.tensor(ious)
    return (ious>=thr).float().mean().item()



# Huấn luyện student (CLIPSeg)
- Mặc định **đóng băng** backbone, fine-tune *decoder + lớp cuối* để tiết kiệm thời gian.  
- Hỗn hợp loss: **BCE + Dice**, nếu có pseudo từ LAVT thì coi như nhãn (hoặc mix với GT).  
- Thêm **early-stopping** đơn giản theo mIoU trên val.


In [ ]:
import os, math, time, numpy as np, torch
import warnings
from torch import nn
from torch.utils.data import DataLoader, ConcatDataset
from tqdm import tqdm
from transformers import AutoProcessor, CLIPSegForImageSegmentation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====== MODEL & PROCESSOR ======
MODEL_ID = "CIDAS/clipseg-rd64-refined"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = CLIPSegForImageSegmentation.from_pretrained(MODEL_ID)

print("unfreeze toàn bộ model")
for name, p in model.named_parameters():
    p.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
model.to(device)

# ====== LOSS / OPTIM (IMPROVED) ======
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return focal_loss.mean()

focal_loss = FocalLoss(alpha=0.25, gamma=2.0)

def dice_loss(logits, target, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs*target).sum(dim=(2,3)) + eps
    den = probs.sum(dim=(2,3)) + target.sum(dim=(2,3)) + eps
    return 1 - (num/den).mean()

optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=LR, weight_decay=WD, betas=(0.9, 0.999))
best_miou = 0.0

def ensure_4d_logits(logits):
    if logits.ndim == 3: logits = logits.unsqueeze(1)
    elif logits.ndim == 4 and logits.shape[1] != 1: logits = logits[:, :1]
    return logits

def to_batch_inputs(batch, processor, device):
    if isinstance(batch, dict):
        texts, images, masks = batch.get("text",[]), batch.get("image",[]), batch.get("mask",[])
        batch = [{"image": images[i], "mask": masks[i], "text": texts[i]} for i in range(len(texts))]
    images  = [b["image"] for b in batch]
    prompts = []
    for b in batch:
        t = b.get("text","")
        if t is None or (isinstance(t,float) and np.isnan(t)): t=""
        prompts.append(str(t))
    text_inputs  = processor.tokenizer(prompts, padding="max_length", truncation=True, return_tensors="pt")
    image_inputs = processor.image_processor(images=images, return_tensors="pt")
    inputs = {**text_inputs, **image_inputs}
    masks = torch.stack([torch.from_numpy(b["mask"]).float().unsqueeze(0) for b in batch], dim=0)
    return {k:v.to(device) for k,v in inputs.items()}, masks.to(device)

# ---- Evaluate có thanh tqdm ----
def evaluate(loader, max_batches=0, desc="Val"):
    model.eval()
    miou_list, p05_list = [], []
    total = max_batches if max_batches and max_batches > 0 else len(loader)
    with torch.no_grad():
        pbar = tqdm(loader, total=total, leave=False, desc=desc, dynamic_ncols=True)
        for bi, batch in enumerate(pbar, start=1):
            inputs, masks = to_batch_inputs(batch, processor, device)
            logits = ensure_4d_logits(model(**inputs).logits)
            target = torch.nn.functional.interpolate(masks, size=logits.shape[-2:], mode="nearest")
            probs = torch.sigmoid(logits).squeeze(1)
            tgt   = target.squeeze(1)

            miou_list.append(iou_bin(probs, tgt))
            p05_list.append(precision_at_threshold(probs, tgt, thr=0.5))

            cur_miou = float(np.mean(miou_list))
            cur_p05  = float(np.mean(p05_list))
            pbar.set_postfix(mIoU=f"{cur_miou:.4f}", P05=f"{cur_p05:.4f}")

            if max_batches and bi >= max_batches:
                break
    return float(np.mean(miou_list)) if miou_list else 0.0, float(np.mean(p05_list)) if p05_list else 0.0

# ====== Mixed Precision & Scheduler (Cosine with Warmup) ======
from torch.cuda.amp import autocast, GradScaler
use_amp = torch.cuda.is_available()
scaler = GradScaler() if use_amp else None

# Dùng tổng bước theo TARGET_STEPS thay vì len(train_loader)
total_steps = TARGET_STEPS

def get_lr(step):
    # step là số bước đã update optimizer (0-based theo LambdaLR)
    if step < WARMUP_STEPS:
        return float(step) / float(max(1, WARMUP_STEPS))  # (0 -> 1)
    else:
        progress = float(step - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
        return 0.5 * (1.0 + math.cos(math.pi * progress))  # (1 -> 0)

from torch.optim.lr_scheduler import LambdaLR
scheduler = LambdaLR(optim, lr_lambda=get_lr)

# ====== Checkpoint helpers ======

global_step = 0

def save_checkpoint(tag="last"):
    state = {
        "model": model.state_dict(),
        "optim": optim.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": (scaler.state_dict() if scaler is not None else None),
        "global_step": global_step,
        "best_miou": best_miou,
        "hparams": {
            "LR": LR, "WD": WD, "TARGET_STEPS": TARGET_STEPS, "WARMUP_STEPS": WARMUP_STEPS,
            "BATCH_SIZE": BATCH_SIZE, "VAL_EVERY_STEPS": VAL_EVERY_STEPS
        }
    }
    path = CKPT_LAST if tag == "last" else CKPT_BEST
    torch.save(state, path)
    if tag == "best":
        # Lưu thêm dạng HuggingFace cho suy luận
        os.makedirs(HF_BEST_DIR, exist_ok=True)
        model.save_pretrained(HF_BEST_DIR)
        processor.save_pretrained(HF_BEST_DIR)

def maybe_resume():
    global global_step, best_miou
    if os.path.exists(CKPT_LAST):
        ckpt = torch.load(CKPT_LAST, map_location=device)
        model.load_state_dict(ckpt["model"])
        optim.load_state_dict(ckpt["optim"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if scaler is not None and ckpt.get("scaler") is not None:
            scaler.load_state_dict(ckpt["scaler"])
        global_step = int(ckpt.get("global_step", 0))
        best_miou   = float(ckpt.get("best_miou", 0.0))
        print(f"Resume thành công từ {CKPT_LAST} @ step={global_step} | best_mIoU={best_miou:.4f}")
    else:
        print("Không tìm thấy checkpoint. Train mới từ đầu.")

print(f"Target steps: {TARGET_STEPS}, Warmup: {WARMUP_STEPS}, Batch size: {BATCH_SIZE}")
maybe_resume()

start_time = time.time()

# ====== TRAIN ======
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_iter = iter(train_loader)
    remaining = max(0, TARGET_STEPS - global_step)
    pbar = tqdm(total=remaining, desc=f"Epoch {epoch}/{EPOCHS} (Steps)", dynamic_ncols=True, mininterval=0.5)
    losses = []

    while global_step < TARGET_STEPS:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        if use_amp:
            with autocast():
                inputs, masks = to_batch_inputs(batch, processor, device)
                logits = ensure_4d_logits(model(**inputs).logits)
                gt = torch.nn.functional.interpolate(masks, size=logits.shape[-2:], mode="nearest")
                loss = focal_loss(logits, gt) + 2.0 * dice_loss(logits, gt)

            optim.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optim)
            scaler.update()
        else:
            inputs, masks = to_batch_inputs(batch, processor, device)
            logits = ensure_4d_logits(model(**inputs).logits)
            gt = torch.nn.functional.interpolate(masks, size=logits.shape[-2:], mode="nearest")
            loss = focal_loss(logits, gt) + 2.0 * dice_loss(logits, gt)

            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optim.step()

        # update step counters & scheduler
        global_step += 1
        scheduler.step()
        losses.append(loss.item())
        current_lr = optim.param_groups[0]['lr']
        pbar.update(1)
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}", step=global_step)

        # LOG định kỳ
        if global_step % LOG_EVERY_STEPS == 0:
            avg_loss = float(np.mean(losses[-LOG_EVERY_STEPS:])) if len(losses) >= LOG_EVERY_STEPS else float(np.mean(losses))
            elapsed = time.time() - start_time
            tqdm.write(f"[E{epoch} step {global_step}] loss={avg_loss:.4f} | lr={current_lr:.2e} | time={elapsed/60:.1f}m")

        # VAL mỗi 500 bước (trên CẢ 2 tập)
        if global_step % VAL_EVERY_STEPS == 0 or global_step == TARGET_STEPS:
            model.eval()
            # RefCOCO
            miou_rcc, p05_rcc = evaluate(val_loader_rcc,
                                         max_batches=(VAL_MAX_BATCHES if VAL_MAX_BATCHES>0 else 0),
                                         desc=f"Val RefCOCO @step {global_step}")
            # RefCOCO+
            miou_rccp, p05_rccp = evaluate(val_loader_rccp,
                                           max_batches=(VAL_MAX_BATCHES if VAL_MAX_BATCHES>0 else 0),
                                           desc=f"Val RefCOCO+ @step {global_step}")

            avg_miou = (miou_rcc + miou_rccp) / 2.0
            tqdm.write(f"[E{epoch} step {global_step}] "
                       f"RefCOCO mIoU={miou_rcc:.4f}|P@0.5={p05_rcc:.4f} | "
                       f"RefCOCO+ mIoU={miou_rccp:.4f}|P@0.5={p05_rccp:.4f} | "
                       f"AVG mIoU={avg_miou:.4f}")

            # Lưu best + autosave last
            if avg_miou > best_miou:
                best_miou = avg_miou
                save_checkpoint(tag="best")
                tqdm.write(f"SAVED BEST: avg_mIoU={best_miou:.4f} (RefCOCO={miou_rcc:.4f}, RefCOCO+={miou_rccp:.4f})")
            # luôn lưu 'last' để resume
            if global_step % CKPT_EVERY_STEPS == 0 or global_step == TARGET_STEPS:
                save_checkpoint(tag="last")
                tqdm.write(f"Saved LAST checkpoint @ step {global_step} → {CKPT_LAST}")
            model.train()

        # Nếu đã đạt TARGET_STEPS thì dừng
        if global_step >= TARGET_STEPS:
            break

    pbar.close()

    # VAL sau epoch
    model.eval()
    miou_rcc, p05_rcc = evaluate(val_loader_rcc, max_batches=0, desc=f"Val RefCOCO E{epoch} (full)")
    miou_rccp, p05_rccp = evaluate(val_loader_rccp, max_batches=0, desc=f"Val RefCOCO+ E{epoch} (full)")
    avg_miou = (miou_rcc + miou_rccp) / 2.0

    print(f"\n{'='*80}")
    print(f"EPOCH {epoch}/{EPOCHS} COMPLETED (đã chạy {global_step}/{TARGET_STEPS} steps):")
    print(f"  RefCOCO:  mIoU={miou_rcc:.4f} | P@0.5={p05_rcc:.4f}")
    print(f"  RefCOCO+: mIoU={miou_rccp:.4f} | P@0.5={p05_rccp:.4f}")
    print(f"  AVERAGE:  mIoU={avg_miou:.4f}")
    print(f"  Best AVG mIoU so far: {best_miou:.4f}")
    print(f"{'='*80}\n")

    # Cập nhật best & lưu cuối epoch
    if avg_miou > best_miou:
        best_miou = avg_miou
        save_checkpoint(tag="best")
        print(f"SAVED BEST at epoch {epoch}: avg_mIoU={best_miou:.4f}")
    save_checkpoint(tag="last")
    print(f"💾 Saved LAST checkpoint at end of epoch → {CKPT_LAST}")

total_time = time.time() - start_time
print(f"\n{'='*80}")
print(f"TRAINING COMPLETED!")
print(f"Total time: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
print(f"Total steps trained: {global_step} / {TARGET_STEPS}")
print(f"Best Average mIoU: {best_miou:.4f} ({best_miou*100:.2f}%)")
print(f"Checkpoints:")
print(f"   - Last: {CKPT_LAST}")
print(f"   - Best (PyTorch): {CKPT_BEST}")
print(f"   - Best (HF weights for inference): {HF_BEST_DIR}")
print(f"{'='*80}\n")


Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

...

unfreeze toàn bộ model
Trainable params: 150,747,746 / 150,747,746 (100.00%)


/tmp/ipython-input-3502111163.py:100: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None


Target steps: 30000, Warmup: 500, Batch size: 18
Resume thành công từ /content/drive/MyDrive/referring-image-segmentation/ckpts/clipseg_student/ckpt_last.pt @ step=30000 | best_mIoU=0.5014


Epoch 1/1 (Steps): 0it [00:00, ?it/s]


KeyboardInterrupt: 